# OTUS | $p p \rightarrow J/\psi \rightarrow \mu^+ \mu^-$


# Load Required Libraries


In [ ]:
%matplotlib inline
import copy
import os
import sys
from pathlib import Path

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")


import csv
import hashlib
import json
import math
import random
import time
from contextlib import contextmanager

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

#-- Add the original utilityFunctions/ directory, as in ppzee.ipynb --#
OTUS_UTILITY_DIR = os.environ.get("OTUS_UTILITY_DIR", "utilityFunctions")
sys.path.append(OTUS_UTILITY_DIR)
from func_utils import data_loss

try:
    from scipy.stats import ks_2samp, wasserstein_distance
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False


# Meta Parameters


In [ ]:
SMOKE_TEST = os.getenv("OTUS_SMOKE_TEST", "0") == "1"

RUN_TRAINING = os.getenv("OTUS_RUN_TRAINING", "1") == "1"
RUN_FINAL_TEST = os.getenv("OTUS_RUN_FINAL_TEST", "0") == "1"

MANUAL_CHECKPOINT_PATH = ""
if not MANUAL_CHECKPOINT_PATH:
    MANUAL_CHECKPOINT_PATH = os.getenv("OTUS_CHECKPOINT_PATH", "").strip()
if MANUAL_CHECKPOINT_PATH:
    RUN_TRAINING = False

DATA_CONFIG = {
    "z_path": os.getenv(
        "OTUS_Z_PATH",
        "/kaggle/input/datasets/viratvarada/jpsi-data/pp_jpsi_mumu_full_aligned.npy",
    ),
    "x_path": os.getenv(
        "OTUS_X_PATH",
        "/kaggle/input/datasets/viratvarada/jpsi-data/pp_jpsi_mumu_cms_real_xspace.npy",
    ),
    "output_dir": os.getenv(
        "OTUS_OUTPUT_DIR",
        "/kaggle/working/jpsi_otus_stochastic_residual_v12"
        if Path("/kaggle/working").exists()
        else "outputs/jpsi_otus_stochastic_residual_v12",
    ),
    # These ranges affect plots only. They are NOT training targets or cuts.
    "mass_plot_range": [2.9, 3.3],
    "mass_bin_width": 0.005,
    "channel_label": "J/psi -> mu+ mu-",
    "column_order": ["mu- px", "mu- py", "mu- pz", "mu- E",
                     "mu+ px", "mu+ py", "mu+ pz", "mu+ E"],
}

In [ ]:
COMMON_TRAINING_CONFIG = {
    "seed": 1701,
    "muon_mass_gev": 0.1056583755,
    "canonicalize_energies": True,
    "split": {"train": 0.80, "validation": 0.10, "test": 0.10},
    "model": {
        "hidden_dims": [512, 512, 512, 512, 512],
        "activation": "SiLU",
        # Residual coordinates are [dlogpt-, deta-, dphi-, dlogpt+, deta+, dphi+].
        "mean_residual_limits": [0.25, 0.30, 0.30, 0.25, 0.30, 0.30],
        # A small nonzero core floor prevents complete stochastic collapse once
        # the core-noise stages begin. Values are dimensionless coordinate units.
        "core_sigma_floors": [0.0010, 0.0005, 0.0005, 0.0010, 0.0005, 0.0005],
        "core_sigma_scales": [0.050, 0.025, 0.025, 0.050, 0.025, 0.025],
        "tail_sigma_scales": [0.150, 0.080, 0.080, 0.150, 0.080, 0.080],
        "core_log_sigma_bias": -3.0,
        "tail_log_sigma_bias": -5.0,
        "student_t_degrees_of_freedom": 4.0,
        "maximum_heavy_noise": 12.0,
    },
    "loader": {
        "batch_size": 4096,
        "steps_per_epoch": 64,
        "validation_size": 32768,
        "validation_draws": 4,
    },
    "loss": {
        "raw_swd": 0.50,
        "raw_marginal_w1": 0.50,
        "physics_swd": 1.00,
        "log_pair_mass_w1": 1.00,
        "log_pair_pt_w1": 1.00,
        "lepton_log_pt_w1": 0.50,
        "lepton_eta_w1": 0.25,
        "delta_phi_w1": 0.25,
        # Equal-weight quantile bands resolve peak, shoulders, and tails
        # without encoding a resonance mass or an absolute mass window.
        "mass_quantile_band_w1": 1.25,
        "mass_quantile_edges": [
            0.000, 0.005, 0.010, 0.020, 0.035,
            0.050, 0.075, 0.100, 0.125, 0.150,
            0.200, 0.250, 0.300, 0.350, 0.400,
            0.450, 0.500,
            0.550, 0.600, 0.650, 0.700, 0.750,
            0.800, 0.850, 0.875, 0.900, 0.925,
            0.950, 0.965, 0.980, 0.990, 0.995,
            1.000,
        ],
        # Differentiable relative CDF matching at generic tail quantiles.
        # Everything is defined in standardized log mass, so no resonance
        # mass scale or channel-specific window enters the objective.
        "mass_tail_cdf": 0.50,
        # Lower-half probabilities; their upper-tail mirrors are automatic.
        "mass_tail_quantiles": [
            0.005, 0.01, 0.02, 0.035, 0.05, 0.075,
            0.10, 0.15, 0.20, 0.30, 0.40,
        ],
        "mass_tail_softness": 0.08,
        # Match the widths of adjacent quantile intervals. This penalizes
        # moving shoulder events into excessively distant tails.
        "mass_quantile_spacing": 0.75,
        "mass_quantile_spacing_epsilon": 1.0e-3,
        # Fixed training-quantile bins directly constrain local probability
        # occupancy. Boundaries are derived from each channel's x/z training
        # sample; this list contains probabilities, not masses.
        "mass_fixed_bin_occupancy": 1.00,
        "mass_fixed_bin_quantile_edges": [
            0.000, 0.010, 0.025, 0.050,
            0.075, 0.100, 0.125, 0.150,
            0.175, 0.200, 0.225, 0.250,
            0.275, 0.300, 0.350, 0.400,
            0.450, 0.500,
            0.550, 0.600, 0.650, 0.700,
            0.725, 0.750, 0.775, 0.800,
            0.825, 0.850, 0.875, 0.900,
            0.925, 0.950, 0.975, 0.990,
            1.000,
        ],
        # Soft CDF transitions are measured in standardized log mass.
        "mass_fixed_bin_softness": 0.020,
        "mass_fixed_bin_huber_beta": 0.050,
        # Cartesian-momentum tail term retained from the previous baseline.
        "tail_w1": 0.10,
        "tail_fraction": 0.10,
        "reconstruction_physics_mse": 1.00,
    },
    "selection_score": {
        "x_sim": 1.00,
        "z_prior": 0.50,
        "x_reconstruction": 0.10,
        "mass_quantile_shape": 0.50,
        # Checkpoint-only penalties use the same dense, scale-free grid.
        "mass_tail_relative_error": 0.25,
        "mass_quantile_spacing": 0.15,
        "mass_fixed_bin_occupancy": 1.00,
    },
    "gradient_clip_norm": 5.0,
    "eval_every": 2,
    "stages": [
        {
            "name": "stage1_residual_otus_warmup",
            "epochs": 60,
            "lr": 1.0e-3,
            "beta": 1.0,
            "lambda_z": 2.0,
            "tau_x": 0.0,
            "nu_encoder": 25.0,
            "nu_decoder": 25.0,
            "num_slices": 256,
            "batch_size": 8192,
            "freeze_encoder": False,
            "freeze_decoder": False,
            "core_noise_multiplier": 0.0,
            "tail_noise_multiplier": 0.0,
            "patience": None,
        },
        {
            "name": "stage2_gaussian_core_transport",
            "epochs": 100,
            "lr": 3.0e-4,
            "beta": 1.0,
            "lambda_z": 1.0,
            "tau_x": 0.5,
            "nu_encoder": 0.25,
            "nu_decoder": 0.25,
            "num_slices": 512,
            "batch_size": 8192,
            "freeze_encoder": False,
            "freeze_decoder": False,
            "core_noise_multiplier": 1.0,
            "tail_noise_multiplier": 0.0,
            "patience": 20,
        },
        {
            "name": "stage3_stochastic_joint_transport",
            "epochs": 120,
            "lr": 1.0e-4,
            "beta": 1.0,
            "lambda_z": 1.0,
            "tau_x": 1.0,
            "nu_encoder": 0.0,
            "nu_decoder": 0.0,
            "num_slices": 768,
            "batch_size": 16384,
            "freeze_encoder": False,
            "freeze_decoder": False,
            "core_noise_multiplier": 1.0,
            "tail_noise_multiplier": 0.25,
            "mass_shape_scale": 0.5,
            "mass_quantile_band_scale": 1.0,
            "patience": 24,
        },
        {
            "name": "stage4_heavy_tail_transport",
            "epochs": 120,
            "lr": 3.0e-5,
            "beta": 1.0,
            "lambda_z": 0.75,
            "tau_x": 1.5,
            "nu_encoder": 0.0,
            "nu_decoder": 0.0,
            "num_slices": 1024,
            "batch_size": 16384,
            "freeze_encoder": False,
            "freeze_decoder": False,
            "core_noise_multiplier": 1.0,
            "tail_noise_multiplier": 1.0,
            "mass_shape_scale": 1.0,
            "mass_quantile_band_scale": 1.0,
            "mass_fixed_bin_occupancy_scale": 0.5,
            "patience": 28,
        },
        {
            "name": "stage5_stochastic_response_polish",
            "epochs": 80,
            "lr": 5.0e-6,
            "beta": 1.0,
            "lambda_z": 0.5,
            "tau_x": 2.0,
            "nu_encoder": 0.0,
            "nu_decoder": 0.0,
            "num_slices": 1024,
            "batch_size": 16384,
            "freeze_encoder": False,
            "freeze_decoder": False,
            "core_noise_multiplier": 1.0,
            "tail_noise_multiplier": 1.0,
            "mass_shape_scale": 1.0,
            "mass_quantile_band_scale": 1.0,
            "mass_fixed_bin_occupancy_scale": 0.75,
            "patience": 30,
        },
    ],
}



In [ ]:
ACTIVE_CONFIG = copy.deepcopy(COMMON_TRAINING_CONFIG)
if SMOKE_TEST:
    ACTIVE_CONFIG["model"]["hidden_dims"] = [64, 64]
    ACTIVE_CONFIG["loader"]["batch_size"] = 256
    ACTIVE_CONFIG["loader"]["steps_per_epoch"] = 4
    ACTIVE_CONFIG["loader"]["validation_size"] = 512
    ACTIVE_CONFIG["loader"]["validation_draws"] = 1
    ACTIVE_CONFIG["eval_every"] = 1
    for stage in ACTIVE_CONFIG["stages"]:
        stage["epochs"] = 1
        stage["num_slices"] = 24
        stage["batch_size"] = ACTIVE_CONFIG["loader"]["batch_size"]
        stage["patience"] = None

if SMOKE_TEST and "OTUS_OUTPUT_DIR" not in os.environ:
    DATA_CONFIG["output_dir"] = DATA_CONFIG["output_dir"] + "_smoke"
OUTPUT_DIR = Path(DATA_CONFIG["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("SMOKE_TEST:", SMOKE_TEST)
print("RUN_TRAINING:", RUN_TRAINING)
print("RUN_FINAL_TEST:", RUN_FINAL_TEST)
print("Checkpoint path:", MANUAL_CHECKPOINT_PATH or "(none; use output directory fallback)")
print("Output directory:", OUTPUT_DIR.resolve())


In [ ]:
def select_device():
    requested = os.getenv("OTUS_DEVICE", "auto").lower()
    if requested != "auto":
        return torch.device(requested)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = select_device()
torch.set_float32_matmul_precision("high")



In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if DEVICE.type == "mps" and hasattr(torch, "mps") and hasattr(torch.mps, "manual_seed"):
        torch.mps.manual_seed(seed)

seed_everything(ACTIVE_CONFIG["seed"])



In [ ]:
COMMON_CONFIG_JSON = json.dumps(COMMON_TRAINING_CONFIG, sort_keys=True, separators=(",", ":"))
COMMON_CONFIG_HASH = hashlib.sha256(COMMON_CONFIG_JSON.encode("utf-8")).hexdigest()

with (OUTPUT_DIR / "common_training_config.json").open("w", encoding="utf-8") as handle:
    json.dump(COMMON_TRAINING_CONFIG, handle, indent=2, sort_keys=True)
with (OUTPUT_DIR / "data_config.json").open("w", encoding="utf-8") as handle:
    json.dump(DATA_CONFIG, handle, indent=2, sort_keys=True)

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Common-config SHA256:", COMMON_CONFIG_HASH)
print("Future Z runs should report the exact same common-config hash.")


# Load Data


In [ ]:
MUON_MASS = float(ACTIVE_CONFIG["muon_mass_gev"])
EPS = 1.0e-8



In [ ]:
def invariant_mass_np(values):
    pair = values[:, 0:4] + values[:, 4:8]
    mass2 = pair[:, 3] ** 2 - np.sum(pair[:, 0:3] ** 2, axis=1)
    return np.sqrt(np.clip(mass2, 0.0, None))



In [ ]:
def canonicalize_energies_np(values, mass=MUON_MASS):
    values = np.asarray(values, dtype=np.float32).copy()
    for start in (0, 4):
        p2 = np.sum(values[:, start:start + 3] ** 2, axis=1)
        values[:, start + 3] = np.sqrt(np.clip(p2 + mass ** 2, 0.0, None))
    return values



In [ ]:
def clean_p4_array(values, label):
    values = np.asarray(values)
    if values.ndim != 2 or values.shape[1] < 8:
        raise ValueError(f"{label} must have shape (N, >=8); got {values.shape}")
    values = values[:, :8].astype(np.float32, copy=False)
    finite = np.isfinite(values).all(axis=1)
    pt1 = np.hypot(values[:, 0], values[:, 1])
    pt2 = np.hypot(values[:, 4], values[:, 5])
    physical = finite & (pt1 > 1.0e-5) & (pt2 > 1.0e-5)
    removed = int((~physical).sum())
    print(f"{label}: removed {removed:,}/{len(values):,} non-finite or zero-pT rows")
    values = values[physical]
    if ACTIVE_CONFIG["canonicalize_energies"]:
        values = canonicalize_energies_np(values)
    return values



In [ ]:
def p4_from_pt_eta_phi_np(pt1, eta1, phi1, pt2, eta2, phi2):
    def one(pt, eta, phi):
        px = pt * np.cos(phi)
        py = pt * np.sin(phi)
        pz = pt * np.sinh(eta)
        energy = np.sqrt(px ** 2 + py ** 2 + pz ** 2 + MUON_MASS ** 2)
        return np.stack([px, py, pz, energy], axis=1)
    return np.concatenate([one(pt1, eta1, phi1), one(pt2, eta2, phi2)], axis=1).astype(np.float32)



In [ ]:
def synthetic_samples(n=8192, seed=123):
    rng = np.random.default_rng(seed)
    pt1 = rng.gamma(3.0, 1.2, n) + 2.0
    pt2 = pt1 * np.exp(rng.normal(0.0, 0.08, n))
    eta1 = rng.uniform(-2.2, 2.2, n)
    eta2 = -eta1 + rng.normal(0.0, 0.20, n)
    phi1 = rng.uniform(-np.pi, np.pi, n)
    phi2 = phi1 + np.pi + rng.normal(0.0, 0.12, n)
    z = p4_from_pt_eta_phi_np(pt1, eta1, phi1, pt2, eta2, phi2)
    x = p4_from_pt_eta_phi_np(
        pt1 * np.exp(rng.normal(0.002, 0.018, n)),
        eta1 + rng.normal(0.0, 0.006, n),
        phi1 + rng.normal(0.0, 0.004, n),
        pt2 * np.exp(rng.normal(-0.001, 0.020, n)),
        eta2 + rng.normal(0.0, 0.006, n),
        phi2 + rng.normal(0.0, 0.004, n),
    )
    return z, x



In [ ]:
def load_samples():
    z_path = Path(DATA_CONFIG["z_path"])
    x_path = Path(DATA_CONFIG["x_path"])
    if SMOKE_TEST:
        print("Using synthetic data for the smoke test.")
        return synthetic_samples()
    missing = [str(path) for path in (z_path, x_path) if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "Missing input file(s):\n  " + "\n  ".join(missing)
            + "\nSet OTUS_Z_PATH and OTUS_X_PATH or edit DATA_CONFIG."
        )
    return np.load(z_path), np.load(x_path)


### Load the CMS and MadGraph arrays


In [ ]:
z_raw, x_raw = load_samples()
z_data = clean_p4_array(z_raw, "Theory z")
x_data = clean_p4_array(x_raw, "Detector x")
del z_raw, x_raw

print("z_data:", z_data.shape, "x_data:", x_data.shape)
print("Assumed order:", DATA_CONFIG["column_order"])


## Define Physics and Conditioning Variables


In [ ]:
def physics_features_np(values):
    p1, p2 = values[:, 0:4], values[:, 4:8]
    pair = p1 + p2
    pt1 = np.hypot(p1[:, 0], p1[:, 1])
    pt2 = np.hypot(p2[:, 0], p2[:, 1])
    pair_pt = np.hypot(pair[:, 0], pair[:, 1])
    eta1 = np.arcsinh(p1[:, 2] / np.clip(pt1, EPS, None))
    eta2 = np.arcsinh(p2[:, 2] / np.clip(pt2, EPS, None))
    phi1 = np.arctan2(p1[:, 1], p1[:, 0])
    phi2 = np.arctan2(p2[:, 1], p2[:, 0])
    delta_phi = np.arctan2(np.sin(phi1 - phi2), np.cos(phi1 - phi2))
    rapidity = 0.5 * np.log(
        np.clip(pair[:, 3] + pair[:, 2], EPS, None)
        / np.clip(pair[:, 3] - pair[:, 2], EPS, None)
    )
    mass = invariant_mass_np(values)
    return np.stack(
        [
            np.log(np.clip(mass, EPS, None)),
            np.log(np.clip(pair_pt, EPS, None)),
            np.log(np.clip(pt1, EPS, None)),
            np.log(np.clip(pt2, EPS, None)),
            eta1,
            eta2,
            rapidity,
            np.cos(delta_phi),
            np.sin(delta_phi),
            eta1 - eta2,
        ],
        axis=1,
    ).astype(np.float32)



In [ ]:
def response_condition_features_np(values):
    p1, p2 = values[:, 0:4], values[:, 4:8]
    pt1 = np.hypot(p1[:, 0], p1[:, 1])
    pt2 = np.hypot(p2[:, 0], p2[:, 1])
    eta1 = np.arcsinh(p1[:, 2] / np.clip(pt1, EPS, None))
    eta2 = np.arcsinh(p2[:, 2] / np.clip(pt2, EPS, None))
    phi1 = np.arctan2(p1[:, 1], p1[:, 0])
    phi2 = np.arctan2(p2[:, 1], p2[:, 0])
    physics = physics_features_np(values)
    return np.stack(
        [
            np.log(np.clip(pt1, EPS, None)), eta1, np.sin(phi1), np.cos(phi1),
            np.log(np.clip(pt2, EPS, None)), eta2, np.sin(phi2), np.cos(phi2),
            physics[:, 0], physics[:, 1], physics[:, 6],
            physics[:, 7], physics[:, 8], physics[:, 9],
        ],
        axis=1,
    ).astype(np.float32)



In [ ]:
RESPONSE_CONDITION_NAMES = [
    "log pT(mu-)", "eta(mu-)", "sin phi(mu-)", "cos phi(mu-)",
    "log pT(mu+)", "eta(mu+)", "sin phi(mu+)", "cos phi(mu+)",
    "log m(mumu)", "log pair pT", "pair rapidity",
    "cos delta-phi", "sin delta-phi", "delta-eta",
]



In [ ]:
PHYSICS_NAMES = [
    "log m(mumu)", "log pair pT", "log pT(mu-)", "log pT(mu+)",
    "eta(mu-)", "eta(mu+)", "pair rapidity", "cos delta-phi",
    "sin delta-phi", "delta-eta",
]



In [ ]:
def report_support(values, label):
    features = physics_features_np(values)
    mass = np.exp(features[:, 0])
    pair_pt = np.exp(features[:, 1])
    quantiles = [0.01, 0.10, 0.50, 0.90, 0.99]
    print(f"\n{label} support (1%, 10%, 50%, 90%, 99%):")
    for name, sample in [
        ("m(mumu) [GeV]", mass),
        ("pair pT [GeV]", pair_pt),
        ("mu- pT [GeV]", np.exp(features[:, 2])),
        ("mu+ pT [GeV]", np.exp(features[:, 3])),
        ("mu- eta", features[:, 4]),
        ("mu+ eta", features[:, 5]),
        ("pair rapidity", features[:, 6]),
    ]:
        print(f"  {name:18s}", np.round(np.quantile(sample, quantiles), 4))
    return features


In [ ]:
z_features_all = report_support(z_data, "Theory z")
x_features_all = report_support(x_data, "Detector x")

z_pair_pt_std = float(np.std(np.exp(z_features_all[:, 1])))
x_pair_pt_std = float(np.std(np.exp(x_features_all[:, 1])))

## Split and Normalize the Data


In [ ]:
def split_indices(n, train_fraction, validation_fraction, seed):
    rng = np.random.default_rng(seed)
    order = rng.permutation(n)
    n_train = int(n * train_fraction)
    n_validation = int(n * validation_fraction)
    return {
        "train": order[:n_train],
        "validation": order[n_train:n_train + n_validation],
        "test": order[n_train + n_validation:],
    }



In [ ]:
split_cfg = ACTIVE_CONFIG["split"]
x_indices = split_indices(
    len(x_data), split_cfg["train"], split_cfg["validation"], ACTIVE_CONFIG["seed"]
)
z_indices = split_indices(
    len(z_data), split_cfg["train"], split_cfg["validation"], ACTIVE_CONFIG["seed"] + 1
)

np.savez_compressed(
    OUTPUT_DIR / "split_indices.npz",
    x_train=x_indices["train"],
    x_validation=x_indices["validation"],
    x_test=x_indices["test"],
    z_train=z_indices["train"],
    z_validation=z_indices["validation"],
    z_test=z_indices["test"],
)

x_train = x_data[x_indices["train"]]
x_validation = x_data[x_indices["validation"]]
x_test = x_data[x_indices["test"]]
z_train = z_data[z_indices["train"]]
z_validation = z_data[z_indices["validation"]]
z_test = z_data[z_indices["test"]]



In [ ]:
def safe_stats(values, name):
    mean = np.mean(values, axis=0).astype(np.float32)
    std = np.std(values, axis=0).astype(np.float32)
    bad = (~np.isfinite(std)) | (std < 1.0e-7)
    if np.any(bad):
        raise RuntimeError(f"{name} has zero/non-finite train std in columns {np.where(bad)[0].tolist()}")
    return mean, std

x_mean, x_std = safe_stats(x_train, "x")
z_mean, z_std = safe_stats(z_train, "z")
x_physics_mean, x_physics_std = safe_stats(physics_features_np(x_train), "x physics")
z_physics_mean, z_physics_std = safe_stats(physics_features_np(z_train), "z physics")



In [ ]:
def response_condition_stats(values):
    features = response_condition_features_np(values)
    mean = np.mean(features, axis=0).astype(np.float32)
    std = np.std(features, axis=0).astype(np.float32)
    if not np.isfinite(mean).all() or not np.isfinite(std).all():
        raise RuntimeError("Non-finite response-condition statistics")
    return mean, np.maximum(std, 1.0e-4).astype(np.float32)

x_condition_mean, x_condition_std = response_condition_stats(x_train)
z_condition_mean, z_condition_std = response_condition_stats(z_train)



In [ ]:
mass_fixed_bin_edges = np.asarray(
    ACTIVE_CONFIG["loss"]["mass_fixed_bin_quantile_edges"], dtype=np.float64
)
if (
    mass_fixed_bin_edges[0] != 0.0
    or mass_fixed_bin_edges[-1] != 1.0
    or np.any(np.diff(mass_fixed_bin_edges) <= 0.0)
):
    raise RuntimeError("mass_fixed_bin_quantile_edges must increase from 0 to 1")


In [ ]:
def fixed_standardized_mass_definition(values, physics_mean, physics_std):
    standardized_log_mass = (
        physics_features_np(values)[:, 0] - physics_mean[0]
    ) / physics_std[0]
    boundaries = np.quantile(
        standardized_log_mass,
        mass_fixed_bin_edges[1:-1],
    ).astype(np.float32)
    softness = float(ACTIVE_CONFIG["loss"]["mass_fixed_bin_softness"])
    soft_cdf = []
    for boundary in boundaries:
        argument = np.clip(
            (boundary - standardized_log_mass) / softness,
            -60.0,
            60.0,
        )
        soft_cdf.append(np.mean(1.0 / (1.0 + np.exp(-argument))))
    target_probabilities = np.diff(
        np.concatenate([[0.0], np.asarray(soft_cdf), [1.0]])
    ).astype(np.float32)
    return boundaries, target_probabilities



In [ ]:
x_fixed_mass_boundaries, x_fixed_mass_targets = fixed_standardized_mass_definition(
    x_train, x_physics_mean, x_physics_std
)
z_fixed_mass_boundaries, z_fixed_mass_targets = fixed_standardized_mass_definition(
    z_train, z_physics_mean, z_physics_std
)

np.savez_compressed(
    OUTPUT_DIR / "fixed_mass_bin_definition.npz",
    quantile_edges=mass_fixed_bin_edges,
    nominal_quantile_probabilities=np.diff(mass_fixed_bin_edges),
    x_soft_target_probabilities=x_fixed_mass_targets,
    z_soft_target_probabilities=z_fixed_mass_targets,
    x_standardized_log_mass_boundaries=x_fixed_mass_boundaries,
    z_standardized_log_mass_boundaries=z_fixed_mass_boundaries,
)

print("\nSplit sizes:")
print("  x:", len(x_train), len(x_validation), len(x_test))
print("  z:", len(z_train), len(z_validation), len(z_test))
print("The test arrays are not used during training or checkpoint selection.")


# Train


## Import Training Specific Libraries and Functions


## Define Meta Network Parameters


Both $E$ and $D$ are stochastic residual maps. Instead of replacing a four-vector directly, each map predicts corrections in $(\log p_T,\eta,\phi)$ for both muons and reconstructs the energies from the muon mass-shell constraint.


## Define Model


In [ ]:
def response_coordinates_torch(values):
    p1, p2 = values[:, 0:4], values[:, 4:8]
    pt1 = torch.sqrt(torch.clamp(p1[:, 0] ** 2 + p1[:, 1] ** 2, min=EPS))
    pt2 = torch.sqrt(torch.clamp(p2[:, 0] ** 2 + p2[:, 1] ** 2, min=EPS))
    eta1 = torch.asinh(p1[:, 2] / pt1)
    eta2 = torch.asinh(p2[:, 2] / pt2)
    phi1 = torch.atan2(p1[:, 1], p1[:, 0])
    phi2 = torch.atan2(p2[:, 1], p2[:, 0])
    return torch.stack(
        [torch.log(pt1), eta1, phi1, torch.log(pt2), eta2, phi2], dim=1
    )



In [ ]:
def response_condition_features_torch(values):
    coordinates = response_coordinates_torch(values)
    physics = physics_features_torch(values)
    logpt1, eta1, phi1, logpt2, eta2, phi2 = coordinates.unbind(dim=1)
    return torch.stack(
        [
            logpt1, eta1, torch.sin(phi1), torch.cos(phi1),
            logpt2, eta2, torch.sin(phi2), torch.cos(phi2),
            physics[:, 0], physics[:, 1], physics[:, 6],
            physics[:, 7], physics[:, 8], physics[:, 9],
        ],
        dim=1,
    )



In [ ]:
def p4_from_response_coordinates_torch(coordinates, muon_mass):
    particles = []
    for start in (0, 3):
        logpt = torch.clamp(coordinates[:, start], min=-7.0, max=9.0)
        eta = torch.clamp(coordinates[:, start + 1], min=-6.0, max=6.0)
        phi = torch.atan2(
            torch.sin(coordinates[:, start + 2]),
            torch.cos(coordinates[:, start + 2]),
        )
        pt = torch.exp(logpt)
        px = pt * torch.cos(phi)
        py = pt * torch.sin(phi)
        pz = pt * torch.sinh(eta)
        energy = torch.sqrt(
            torch.clamp(px.square() + py.square() + pz.square() + muon_mass ** 2, min=EPS)
        )
        particles.append(torch.stack([px, py, pz, energy], dim=1))
    return torch.cat(particles, dim=1)



In [ ]:
def make_response_backbone(input_dim, hidden_dims, activation):
    layers = []
    previous = input_dim
    for width in hidden_dims:
        layers.extend([nn.Linear(previous, width), nn.LayerNorm(width), activation()])
        previous = width
    return nn.Sequential(*layers), previous



In [ ]:
class PhysicalStochasticResidualMap(nn.Module):
    def __init__(self, condition_mean, condition_std, model_config, muon_mass):
        super().__init__()
        self.muon_mass = float(muon_mass)
        self.student_t_df = float(model_config["student_t_degrees_of_freedom"])
        self.maximum_heavy_noise = float(model_config["maximum_heavy_noise"])
        if self.student_t_df <= 2.0:
            raise ValueError("Student-t degrees of freedom must exceed two")

        self.register_buffer(
            "condition_mean", torch.as_tensor(condition_mean, dtype=torch.float32)
        )
        self.register_buffer(
            "condition_std", torch.as_tensor(condition_std, dtype=torch.float32)
        )
        for name in ["mean_residual_limits", "core_sigma_floors", "core_sigma_scales", "tail_sigma_scales"]:
            values = torch.as_tensor(model_config[name], dtype=torch.float32)
            if values.numel() != 6:
                raise ValueError(f"{name} must contain six coordinate values")
            self.register_buffer(name, values)

        activation = getattr(nn, model_config["activation"])
        self.backbone, width = make_response_backbone(
            len(condition_mean), model_config["hidden_dims"], activation
        )
        self.response_head = nn.Linear(width, 18)
        nn.init.normal_(self.response_head.weight, mean=0.0, std=1.0e-4)
        with torch.no_grad():
            self.response_head.bias[:6].zero_()
            self.response_head.bias[6:12].fill_(float(model_config["core_log_sigma_bias"]))
            self.response_head.bias[12:18].fill_(float(model_config["tail_log_sigma_bias"]))

        self.core_noise_multiplier = 1.0
        self.tail_noise_multiplier = 1.0

    def set_noise_multipliers(self, core, tail):
        self.core_noise_multiplier = float(core)
        self.tail_noise_multiplier = float(tail)

    def forward(self, raw_input):
        condition = response_condition_features_torch(raw_input)
        standardized = (condition - self.condition_mean) / self.condition_std
        response = self.response_head(self.backbone(standardized))
        mean_raw, core_raw, tail_raw = response.split(6, dim=1)

        mean_delta = torch.tanh(mean_raw) * self.mean_residual_limits
        core_sigma = self.core_sigma_floors + F.softplus(core_raw) * self.core_sigma_scales
        tail_sigma = F.softplus(tail_raw) * self.tail_sigma_scales

        core_noise = torch.randn_like(core_sigma)
        concentration = core_sigma.new_tensor(self.student_t_df / 2.0)
        rate = core_sigma.new_tensor(self.student_t_df / 2.0)
        inverse_scale = torch.distributions.Gamma(concentration, rate).rsample(core_sigma.shape)
        heavy_noise = torch.randn_like(tail_sigma) / torch.sqrt(inverse_scale.clamp_min(1.0e-5))
        heavy_noise = torch.clamp(
            heavy_noise, min=-self.maximum_heavy_noise, max=self.maximum_heavy_noise
        )

        delta = (
            mean_delta
            + self.core_noise_multiplier * core_sigma * core_noise
            + self.tail_noise_multiplier * tail_sigma * heavy_noise
        )
        output_coordinates = response_coordinates_torch(raw_input) + delta
        return p4_from_response_coordinates_torch(output_coordinates, self.muon_mass)



In [ ]:
class DimuonOTUS(nn.Module):
    def __init__(self, x_condition_stats, z_condition_stats, config):
        super().__init__()
        model_config = config["model"]
        muon_mass = config["muon_mass_gev"]
        self.encoder = PhysicalStochasticResidualMap(
            x_condition_stats[0], x_condition_stats[1], model_config, muon_mass
        )
        self.decoder = PhysicalStochasticResidualMap(
            z_condition_stats[0], z_condition_stats[1], model_config, muon_mass
        )

    def set_noise_multipliers(self, core, tail):
        self.encoder.set_noise_multipliers(core, tail)
        self.decoder.set_noise_multipliers(core, tail)

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        z_encoded = self.encode(x)
        return z_encoded, self.decode(z_encoded)



In [ ]:
def set_trainable(model, encoder=True, decoder=True):
    for parameter in model.encoder.parameters():
        parameter.requires_grad_(encoder)
    for parameter in model.decoder.parameters():
        parameter.requires_grad_(decoder)


## Define Loss Functions


In [ ]:
def physics_features_torch(values):
    p1, p2 = values[:, 0:4], values[:, 4:8]
    pair = p1 + p2
    pt1 = torch.sqrt(torch.clamp(p1[:, 0] ** 2 + p1[:, 1] ** 2, min=EPS))
    pt2 = torch.sqrt(torch.clamp(p2[:, 0] ** 2 + p2[:, 1] ** 2, min=EPS))
    pair_pt = torch.sqrt(torch.clamp(pair[:, 0] ** 2 + pair[:, 1] ** 2, min=EPS))
    eta1 = torch.asinh(p1[:, 2] / pt1)
    eta2 = torch.asinh(p2[:, 2] / pt2)
    phi1 = torch.atan2(p1[:, 1], p1[:, 0])
    phi2 = torch.atan2(p2[:, 1], p2[:, 0])
    delta_phi = torch.atan2(torch.sin(phi1 - phi2), torch.cos(phi1 - phi2))
    mass2 = pair[:, 3] ** 2 - torch.sum(pair[:, 0:3] ** 2, dim=1)
    mass = torch.sqrt(torch.clamp(mass2, min=EPS))
    rapidity = 0.5 * torch.log(
        torch.clamp(pair[:, 3] + pair[:, 2], min=EPS)
        / torch.clamp(pair[:, 3] - pair[:, 2], min=EPS)
    )
    return torch.stack(
        [
            torch.log(torch.clamp(mass, min=EPS)),
            torch.log(torch.clamp(pair_pt, min=EPS)),
            torch.log(torch.clamp(pt1, min=EPS)),
            torch.log(torch.clamp(pt2, min=EPS)),
            eta1,
            eta2,
            rapidity,
            torch.cos(delta_phi),
            torch.sin(delta_phi),
            eta1 - eta2,
        ],
        dim=1,
    )



In [ ]:
def sliced_wasserstein(a, b, num_slices, p=2):
    if a.ndim != 2 or b.ndim != 2 or a.shape != b.shape:
        raise ValueError(f"SWD requires equal rank-2 batches; got {a.shape} and {b.shape}")
    theta = torch.randn(
        a.shape[1], int(num_slices), dtype=a.dtype, device=a.device
    )
    theta = theta / torch.clamp(torch.linalg.vector_norm(theta, dim=0, keepdim=True), min=EPS)
    a_sorted = torch.sort(a @ theta, dim=0).values
    b_sorted = torch.sort(b @ theta, dim=0).values
    difference = a_sorted - b_sorted
    if p == 1:
        return difference.abs().mean()
    if p == 2:
        return difference.square().mean()
    raise ValueError("Only p=1 and p=2 are supported")



In [ ]:
def sorted_w1(a, b):
    a = a.reshape(-1)
    b = b.reshape(-1)
    if a.numel() != b.numel():
        raise ValueError("Training W1 requires equal sample counts")
    return (torch.sort(a).values - torch.sort(b).values).abs().mean()



In [ ]:
def direction_anchor(a, b):
    losses = []
    for start in (0, 4):
        losses.append(
            1.0 - F.cosine_similarity(
                a[:, start:start + 3],
                b[:, start:start + 3],
                dim=1,
                eps=1.0e-8,
            )
        )
    return torch.cat(losses).mean()



In [ ]:
class FeatureOTLoss(nn.Module):
    def __init__(
        self,
        raw_mean,
        raw_std,
        physics_mean,
        physics_std,
        loss_config,
        fixed_mass_boundaries,
        fixed_mass_target_probabilities,
    ):
        super().__init__()
        self.register_buffer("raw_mean", torch.as_tensor(raw_mean, dtype=torch.float32))
        self.register_buffer("raw_std", torch.as_tensor(raw_std, dtype=torch.float32))
        self.register_buffer("physics_mean", torch.as_tensor(physics_mean, dtype=torch.float32))
        self.register_buffer("physics_std", torch.as_tensor(physics_std, dtype=torch.float32))
        self.weights = dict(loss_config)
        self.register_buffer(
            "fixed_mass_boundaries",
            torch.as_tensor(fixed_mass_boundaries, dtype=torch.float32),
        )
        self.register_buffer(
            "fixed_mass_target_probabilities",
            torch.as_tensor(fixed_mass_target_probabilities, dtype=torch.float32),
        )
        if self.fixed_mass_boundaries.numel() + 1 != self.fixed_mass_target_probabilities.numel():
            raise ValueError("Fixed mass boundaries/target probabilities have incompatible sizes")
        if not torch.isclose(
            self.fixed_mass_target_probabilities.sum(),
            self.fixed_mass_target_probabilities.new_tensor(1.0),
            atol=1.0e-5,
        ):
            raise ValueError("Fixed mass target probabilities must sum to one")
        self.mass_fixed_bin_softness = float(loss_config["mass_fixed_bin_softness"])
        self.mass_fixed_bin_huber_beta = float(loss_config["mass_fixed_bin_huber_beta"])
        if self.mass_fixed_bin_softness <= 0.0 or self.mass_fixed_bin_huber_beta <= 0.0:
            raise ValueError("Fixed-bin softness and Huber beta must be positive")
        self.tail_fraction = float(loss_config["tail_fraction"])
        self.mass_quantile_edges = tuple(float(q) for q in loss_config["mass_quantile_edges"])
        if self.mass_quantile_edges[0] != 0.0 or self.mass_quantile_edges[-1] != 1.0:
            raise ValueError("mass_quantile_edges must start at 0 and end at 1")
        if any(b <= a for a, b in zip(self.mass_quantile_edges[:-1], self.mass_quantile_edges[1:])):
            raise ValueError("mass_quantile_edges must be strictly increasing")
        self.mass_tail_quantiles = tuple(
            float(q) for q in loss_config["mass_tail_quantiles"]
        )
        if any(q <= 0.0 or q >= 0.5 for q in self.mass_tail_quantiles):
            raise ValueError("mass_tail_quantiles must lie between 0 and 0.5")
        self.mass_tail_softness = float(loss_config["mass_tail_softness"])
        if self.mass_tail_softness <= 0.0:
            raise ValueError("mass_tail_softness must be positive")
        self.mass_quantile_spacing_epsilon = float(
            loss_config["mass_quantile_spacing_epsilon"]
        )
        if self.mass_quantile_spacing_epsilon <= 0.0:
            raise ValueError("mass_quantile_spacing_epsilon must be positive")
        dense_grid = (
            list(self.mass_tail_quantiles)
            + [0.5]
            + [1.0 - q for q in reversed(self.mass_tail_quantiles)]
        )
        self.register_buffer(
            "mass_shape_quantile_grid",
            torch.tensor(dense_grid, dtype=torch.float32),
        )

    def standardized_raw(self, values):
        return (values - self.raw_mean) / self.raw_std

    def standardized_physics(self, values):
        return (physics_features_torch(values) - self.physics_mean) / self.physics_std

    def paired_reconstruction(self, truth, prediction):
        raw_loss = data_loss(
            self.standardized_raw(truth),
            self.standardized_raw(prediction),
            p=2,
        )
        physics_loss = data_loss(
            self.standardized_physics(truth),
            self.standardized_physics(prediction),
            p=2,
        )
        return raw_loss + float(self.weights["reconstruction_physics_mse"]) * physics_loss

    def forward(
        self,
        truth,
        prediction,
        num_slices,
        mass_shape_scale=0.0,
        mass_quantile_band_scale=1.0,
        mass_fixed_bin_occupancy_scale=0.0,
    ):
        raw_truth = self.standardized_raw(truth)
        raw_prediction = self.standardized_raw(prediction)
        physics_truth = self.standardized_physics(truth)
        physics_prediction = self.standardized_physics(prediction)

        raw_marginal = torch.stack(
            [sorted_w1(raw_truth[:, i], raw_prediction[:, i]) for i in range(raw_truth.shape[1])]
        ).mean()

        start = max(0, int((1.0 - self.tail_fraction) * raw_truth.shape[0]))
        tail_terms = []
        for i in [0, 1, 2, 4, 5, 6]:
            truth_tail = torch.sort(raw_truth[:, i].abs()).values[start:]
            prediction_tail = torch.sort(raw_prediction[:, i].abs()).values[start:]
            tail_terms.append((truth_tail - prediction_tail).abs().mean())
        tail_loss = torch.stack(tail_terms).mean()

        # Compare sorted standardized log mass in multiple quantile bands.
        # Each band contributes equally, so the 1--5% tails and 5--20%
        # shoulders are not overwhelmed by the high-density peak.
        mass_truth_sorted = torch.sort(physics_truth[:, 0]).values
        mass_prediction_sorted = torch.sort(physics_prediction[:, 0]).values
        mass_count = mass_truth_sorted.shape[0]
        mass_band_terms = []
        for q_low, q_high in zip(
            self.mass_quantile_edges[:-1], self.mass_quantile_edges[1:]
        ):
            start = min(mass_count - 1, int(q_low * mass_count))
            stop = min(mass_count, max(start + 1, int(q_high * mass_count)))
            mass_band_terms.append(
                (
                    mass_truth_sorted[start:stop]
                    - mass_prediction_sorted[start:stop]
                ).abs().mean()
            )
        mass_quantile_band_w1 = torch.stack(mass_band_terms).mean()

        # Soft empirical-CDF matching. Thresholds and targets come from the
        # current truth batch. Only prediction occupancies receive gradients.
        standardized_mass_truth = physics_truth[:, 0]
        standardized_mass_prediction = physics_prediction[:, 0]
        mass_tail_cdf_terms = []
        softness = self.mass_tail_softness
        for quantile in self.mass_tail_quantiles:
            lower_threshold = torch.quantile(
                standardized_mass_truth.detach(), quantile
            )
            upper_threshold = torch.quantile(
                standardized_mass_truth.detach(), 1.0 - quantile
            )
            truth_lower = torch.sigmoid(
                (lower_threshold - standardized_mass_truth.detach()) / softness
            ).mean()
            prediction_lower = torch.sigmoid(
                (lower_threshold - standardized_mass_prediction) / softness
            ).mean()
            truth_upper = torch.sigmoid(
                (standardized_mass_truth.detach() - upper_threshold) / softness
            ).mean()
            prediction_upper = torch.sigmoid(
                (standardized_mass_prediction - upper_threshold) / softness
            ).mean()
            mass_tail_cdf_terms.extend(
                [
                    (prediction_lower - truth_lower).abs()
                    / torch.clamp(truth_lower, min=0.01),
                    (prediction_upper - truth_upper).abs()
                    / torch.clamp(truth_upper, min=0.01),
                ]
            )
        mass_tail_cdf = torch.stack(mass_tail_cdf_terms).mean()

        dense_mass_truth_quantiles = torch.quantile(
            standardized_mass_truth.detach(), self.mass_shape_quantile_grid
        )
        dense_mass_prediction_quantiles = torch.quantile(
            standardized_mass_prediction, self.mass_shape_quantile_grid
        )
        epsilon = self.mass_quantile_spacing_epsilon
        truth_log_spacing = torch.log(
            torch.diff(dense_mass_truth_quantiles).clamp_min(epsilon)
        )
        prediction_log_spacing = torch.log(
            torch.diff(dense_mass_prediction_quantiles).clamp_min(epsilon)
        )
        mass_quantile_spacing = F.smooth_l1_loss(
            prediction_log_spacing,
            truth_log_spacing,
            reduction="mean",
        )

        # Fixed, train-derived soft histogram. Unlike the earlier batchwise
        # CDF thresholds, these boundaries do not jitter between minibatches.
        boundary_cdf = torch.sigmoid(
            (
                self.fixed_mass_boundaries[:, None]
                - standardized_mass_prediction[None, :]
            )
            / self.mass_fixed_bin_softness
        ).mean(dim=1)
        predicted_bin_probabilities = torch.diff(
            torch.cat(
                [
                    boundary_cdf.new_zeros(1),
                    boundary_cdf,
                    boundary_cdf.new_ones(1),
                ]
            )
        )
        relative_bin_error = (
            predicted_bin_probabilities - self.fixed_mass_target_probabilities
        ) / self.fixed_mass_target_probabilities.clamp_min(0.01)
        mass_fixed_bin_occupancy = F.smooth_l1_loss(
            relative_bin_error,
            torch.zeros_like(relative_bin_error),
            beta=self.mass_fixed_bin_huber_beta,
            reduction="mean",
        )

        components = {
            "raw_swd": sliced_wasserstein(raw_truth, raw_prediction, num_slices, p=2),
            "raw_marginal_w1": raw_marginal,
            "physics_swd": sliced_wasserstein(
                physics_truth, physics_prediction, num_slices, p=2
            ),
            "log_pair_mass_w1": sorted_w1(physics_truth[:, 0], physics_prediction[:, 0]),
            "mass_quantile_band_w1": mass_quantile_band_w1,
            "mass_tail_cdf": mass_tail_cdf,
            "mass_quantile_spacing": mass_quantile_spacing,
            "mass_fixed_bin_occupancy": mass_fixed_bin_occupancy,
            "log_pair_pt_w1": sorted_w1(physics_truth[:, 1], physics_prediction[:, 1]),
            "lepton_log_pt_w1": 0.5 * (
                sorted_w1(physics_truth[:, 2], physics_prediction[:, 2])
                + sorted_w1(physics_truth[:, 3], physics_prediction[:, 3])
            ),
            "lepton_eta_w1": 0.5 * (
                sorted_w1(physics_truth[:, 4], physics_prediction[:, 4])
                + sorted_w1(physics_truth[:, 5], physics_prediction[:, 5])
            ),
            "delta_phi_w1": 0.5 * (
                sorted_w1(physics_truth[:, 7], physics_prediction[:, 7])
                + sorted_w1(physics_truth[:, 8], physics_prediction[:, 8])
            ),
            "tail_w1": tail_loss,
        }

        total = truth.new_tensor(0.0)
        for name, value in components.items():
            if name == "mass_quantile_band_w1":
                component_scale = float(mass_quantile_band_scale)
            elif name in {"mass_tail_cdf", "mass_quantile_spacing"}:
                component_scale = float(mass_shape_scale)
            elif name == "mass_fixed_bin_occupancy":
                component_scale = float(mass_fixed_bin_occupancy_scale)
            else:
                component_scale = 1.0
            total = total + component_scale * float(self.weights[name]) * value
        return total, components


## Define Training and Validation Functions


In [ ]:
class RandomBatchSource:
    def __init__(self, array, pin_memory=False):
        tensor = torch.from_numpy(np.ascontiguousarray(array, dtype=np.float32))
        self.tensor = tensor.pin_memory() if pin_memory else tensor
        self.size = len(tensor)

    def draw(self, batch_size, generator):
        indices = torch.randint(0, self.size, (batch_size,), generator=generator)
        return self.tensor.index_select(0, indices).to(
            DEVICE, non_blocking=(DEVICE.type == "cuda")
        )



In [ ]:
def fixed_subset(array, size, seed):
    rng = np.random.default_rng(seed)
    size = min(int(size), len(array))
    indices = rng.choice(len(array), size=size, replace=False)
    return torch.from_numpy(np.ascontiguousarray(array[indices], dtype=np.float32)).to(DEVICE)



In [ ]:
@contextmanager
def fixed_evaluation_rng(seed):
    cpu_state = torch.random.get_rng_state()
    python_state = random.getstate()
    numpy_state = np.random.get_state()
    cuda_state = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    mps_state = None
    if DEVICE.type == "mps" and hasattr(torch.mps, "get_rng_state"):
        mps_state = torch.mps.get_rng_state()
    seed_everything(seed)
    try:
        yield
    finally:
        torch.random.set_rng_state(cpu_state)
        random.setstate(python_state)
        np.random.set_state(numpy_state)
        if cuda_state is not None:
            torch.cuda.set_rng_state_all(cuda_state)
        if mps_state is not None and hasattr(torch.mps, "set_rng_state"):
            torch.mps.set_rng_state(mps_state)



In [ ]:
def average_dicts(rows):
    keys = sorted(set().union(*(row.keys() for row in rows)))
    return {key: float(np.mean([row[key] for row in rows if key in row])) for key in keys}



In [ ]:
def train_epoch(
    model,
    optimizer,
    x_source,
    z_source,
    x_loss_function,
    z_loss_function,
    stage,
    epoch,
):
    model.train()
    batch_size = int(
        stage.get("batch_size", ACTIVE_CONFIG["loader"]["batch_size"])
    )
    # A fixed number of independently resampled batches makes runtime and the
    # meaning of an epoch identical for J/psi and Z, even when sample sizes differ.
    steps = int(ACTIVE_CONFIG["loader"]["steps_per_epoch"])
    generator = torch.Generator(device="cpu")
    generator.manual_seed(ACTIVE_CONFIG["seed"] + 10000 * epoch)
    rows = []

    for _ in range(steps):
        x = x_source.draw(batch_size, generator)
        z = z_source.draw(batch_size, generator)
        optimizer.zero_grad(set_to_none=True)

        z_encoded = model.encode(x)
        x_reconstructed = model.decode(z_encoded)
        reconstruction_loss = x_loss_function.paired_reconstruction(x, x_reconstructed)
        mass_shape_scale = float(stage.get("mass_shape_scale", 0.0))
        mass_quantile_band_scale = float(
            stage.get("mass_quantile_band_scale", 1.0)
        )
        mass_fixed_bin_occupancy_scale = float(
            stage.get("mass_fixed_bin_occupancy_scale", 0.0)
        )
        if stage["lambda_z"] > 0:
            z_prior_loss, _ = z_loss_function(
                z, z_encoded, stage["num_slices"], mass_shape_scale,
                mass_quantile_band_scale,
                mass_fixed_bin_occupancy_scale,
            )
        else:
            # The encoder is frozen in the final refinement. Avoid an
            # expensive latent loss that would be multiplied by zero.
            z_prior_loss = x.new_tensor(0.0)

        need_direct_decode = stage["tau_x"] > 0 or stage["nu_decoder"] > 0
        if need_direct_decode:
            x_from_z = model.decode(z)
            x_sim_loss, _ = x_loss_function(
                x, x_from_z, stage["num_slices"], mass_shape_scale,
                mass_quantile_band_scale,
                mass_fixed_bin_occupancy_scale,
            )
            decoder_anchor = direction_anchor(z, x_from_z)
        else:
            x_sim_loss = x.new_tensor(0.0)
            decoder_anchor = x.new_tensor(0.0)

        encoder_anchor = (
            direction_anchor(x, z_encoded)
            if stage["nu_encoder"] > 0
            else x.new_tensor(0.0)
        )

        total = (
            stage["beta"] * reconstruction_loss
            + stage["lambda_z"] * z_prior_loss
            + stage["tau_x"] * x_sim_loss
            + stage["nu_encoder"] * encoder_anchor
            + stage["nu_decoder"] * decoder_anchor
        )
        if not torch.isfinite(total):
            raise FloatingPointError(f"Non-finite training loss in {stage['name']}")

        total.backward()
        gradient_norm = torch.nn.utils.clip_grad_norm_(
            [parameter for parameter in model.parameters() if parameter.requires_grad],
            max_norm=float(ACTIVE_CONFIG["gradient_clip_norm"]),
        )
        if not torch.isfinite(gradient_norm):
            raise FloatingPointError(f"Non-finite gradient in {stage['name']}")
        optimizer.step()

        rows.append(
            {
                "loss": float(total.detach().cpu()),
                "x_reconstruction": float(reconstruction_loss.detach().cpu()),
                "z_prior": float(z_prior_loss.detach().cpu()),
                "x_sim": float(x_sim_loss.detach().cpu()),
                "encoder_anchor": float(encoder_anchor.detach().cpu()),
                "decoder_anchor": float(decoder_anchor.detach().cpu()),
                "gradient_norm": float(gradient_norm.detach().cpu()),
            }
        )
    return average_dicts(rows)



In [ ]:
@torch.inference_mode()
def evaluate_model(model, x_fixed, z_fixed, x_loss_function, z_loss_function, num_slices):
    model.eval()
    draw_rows = []
    with fixed_evaluation_rng(ACTIVE_CONFIG["seed"] + 424242):
        for _ in range(int(ACTIVE_CONFIG["loader"]["validation_draws"])):
            z_encoded = model.encode(x_fixed)
            x_reconstructed = model.decode(z_encoded)
            x_from_z = model.decode(z_fixed)
            reconstruction = x_loss_function.paired_reconstruction(x_fixed, x_reconstructed)
            z_prior, z_components = z_loss_function(z_fixed, z_encoded, num_slices)
            x_sim, x_components = x_loss_function(x_fixed, x_from_z, num_slices)

            x_mass = torch.exp(physics_features_torch(x_fixed)[:, 0])
            simulated_mass = torch.exp(physics_features_torch(x_from_z)[:, 0])
            mass_q05, mass_q95 = torch.quantile(
                x_mass, x_mass.new_tensor([0.05, 0.95])
            )
            simulated_low_fraction = (simulated_mass < mass_q05).float().mean()
            simulated_high_fraction = (simulated_mass > mass_q95).float().mean()
            mass_tail_fraction_error = (
                (simulated_low_fraction - 0.05).abs()
                + (simulated_high_fraction - 0.05).abs()
            )
            # Equal relative importance across the complete dense tail grid.
            # This is validation-only and therefore need not be differentiable.
            relative_tail_errors = []
            for quantile in ACTIVE_CONFIG["loss"]["mass_tail_quantiles"]:
                lower_threshold, upper_threshold = torch.quantile(
                    x_mass, x_mass.new_tensor([quantile, 1.0 - quantile])
                )
                predicted_lower = (simulated_mass < lower_threshold).float().mean()
                predicted_upper = (simulated_mass > upper_threshold).float().mean()
                relative_tail_errors.append(
                    0.5
                    * (
                        (predicted_lower - quantile).abs()
                        + (predicted_upper - quantile).abs()
                    )
                    / quantile
                )
            mass_tail_relative_error = torch.stack(relative_tail_errors).mean()

            score_cfg = ACTIVE_CONFIG["selection_score"]
            score = (
                score_cfg["x_sim"] * x_sim
                + score_cfg["z_prior"] * z_prior
                + score_cfg["x_reconstruction"] * reconstruction
                + score_cfg["mass_quantile_shape"]
                * x_components["mass_quantile_band_w1"]
                + score_cfg["mass_tail_relative_error"]
                * mass_tail_relative_error
                + score_cfg["mass_quantile_spacing"]
                * x_components["mass_quantile_spacing"]
                + score_cfg["mass_fixed_bin_occupancy"]
                * x_components["mass_fixed_bin_occupancy"]
            )
            draw_rows.append(
                {
                    "score": float(score.cpu()),
                    "x_reconstruction": float(reconstruction.cpu()),
                    "z_prior": float(z_prior.cpu()),
                    "x_sim": float(x_sim.cpu()),
                    "mass_low_fraction": float(simulated_low_fraction.cpu()),
                    "mass_high_fraction": float(simulated_high_fraction.cpu()),
                    "mass_tail_fraction_error": float(mass_tail_fraction_error.cpu()),
                    "mass_tail_relative_error": float(
                        mass_tail_relative_error.cpu()
                    ),
                    "mass_quantile_shape": float(
                        x_components["mass_quantile_band_w1"].cpu()
                    ),
                    "mass_tail_cdf": float(
                        x_components["mass_tail_cdf"].cpu()
                    ),
                    "mass_quantile_spacing": float(
                        x_components["mass_quantile_spacing"].cpu()
                    ),
                    "mass_fixed_bin_occupancy": float(
                        x_components["mass_fixed_bin_occupancy"].cpu()
                    ),
                }
            )
    return average_dicts(draw_rows)



In [ ]:
def checkpoint_payload(model, optimizer, stage, stage_index, epoch, validation, history):
    payload = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
        "common_training_config": COMMON_TRAINING_CONFIG,
        "active_training_config": ACTIVE_CONFIG,
        "common_config_sha256": COMMON_CONFIG_HASH,
        "data_config": DATA_CONFIG,
        "stage": stage,
        "stage_index": int(stage_index),
        "epoch": int(epoch),
        "validation": validation,
        "history": history,
        "numpy_random_state": np.random.get_state(),
        "python_random_state": random.getstate(),
        "torch_random_state": torch.random.get_rng_state(),
    }
    if torch.cuda.is_available():
        payload["cuda_random_state"] = torch.cuda.get_rng_state_all()
    return payload



In [ ]:
def load_checkpoint(path):
    # This notebook creates the checkpoint itself, so loading the full trusted
    # research state (config, optimizer, RNG, and weights) is intentional.
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:  # Compatibility with older PyTorch releases.
        return torch.load(path, map_location=DEVICE)



In [ ]:
def save_checkpoint(path, model, optimizer, stage, stage_index, epoch, validation, history):
    torch.save(
        checkpoint_payload(model, optimizer, stage, stage_index, epoch, validation, history),
        path,
    )



In [ ]:
def write_history(history):
    with (OUTPUT_DIR / "history.json").open("w", encoding="utf-8") as handle:
        json.dump(history, handle, indent=2)
    if history:
        keys = sorted(set().union(*(row.keys() for row in history)))
        with (OUTPUT_DIR / "history.csv").open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=keys)
            writer.writeheader()
            writer.writerows(history)


## Define Model and Hyperparameters


This instantiates the encoder, decoder, detector-space loss, and latent-space loss using the training-only statistics defined above.


In [ ]:
model = DimuonOTUS(
    x_condition_stats=(x_condition_mean, x_condition_std),
    z_condition_stats=(z_condition_mean, z_condition_std),
    config=ACTIVE_CONFIG,
).to(DEVICE)

x_loss_function = FeatureOTLoss(
    x_mean,
    x_std,
    x_physics_mean,
    x_physics_std,
    ACTIVE_CONFIG["loss"],
    x_fixed_mass_boundaries,
    x_fixed_mass_targets,
).to(DEVICE)
z_loss_function = FeatureOTLoss(
    z_mean,
    z_std,
    z_physics_mean,
    z_physics_std,
    ACTIVE_CONFIG["loss"],
    z_fixed_mass_boundaries,
    z_fixed_mass_targets,
).to(DEVICE)

parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(model)
print(f"Trainable parameters: {parameter_count:,}")



In [ ]:
pin_memory = DEVICE.type == "cuda"
x_source = RandomBatchSource(x_train, pin_memory=pin_memory)
z_source = RandomBatchSource(z_train, pin_memory=pin_memory)
validation_size = ACTIVE_CONFIG["loader"]["validation_size"]
common_validation_size = min(validation_size, len(x_validation), len(z_validation))
x_validation_fixed = fixed_subset(
    x_validation, common_validation_size, ACTIVE_CONFIG["seed"] + 20
)
z_validation_fixed = fixed_subset(
    z_validation, common_validation_size, ACTIVE_CONFIG["seed"] + 21
)

# Gradient and mass-shell preflight without taking an optimizer step.
preflight_batch = min(256, ACTIVE_CONFIG["loader"]["batch_size"])
preflight_generator = torch.Generator(device="cpu").manual_seed(ACTIVE_CONFIG["seed"] + 99)
x_preflight = x_source.draw(preflight_batch, preflight_generator)
z_preflight = z_source.draw(preflight_batch, preflight_generator)
model.zero_grad(set_to_none=True)
z_preflight_encoded = model.encode(x_preflight)
x_preflight_reconstructed = model.decode(z_preflight_encoded)
z_preflight_loss, _ = z_loss_function(z_preflight, z_preflight_encoded, 16)
preflight_loss = (
    x_loss_function.paired_reconstruction(x_preflight, x_preflight_reconstructed)
    + z_preflight_loss
)
preflight_loss.backward()

encoder_gradient = sum(
    float(parameter.grad.abs().sum().detach().cpu())
    for parameter in model.encoder.parameters()
    if parameter.grad is not None
)
decoder_gradient = sum(
    float(parameter.grad.abs().sum().detach().cpu())
    for parameter in model.decoder.parameters()
    if parameter.grad is not None
)
model.zero_grad(set_to_none=True)
x_preflight_from_z = model.decode(z_preflight)
direct_x_preflight_loss, _ = x_loss_function(
    x_preflight, x_preflight_from_z, 16
)
direct_x_preflight_loss.backward()
direct_decoder_gradient = sum(
    float(parameter.grad.abs().sum().detach().cpu())
    for parameter in model.decoder.parameters()
    if parameter.grad is not None
)
model.zero_grad(set_to_none=True)

if (
    not np.isfinite(encoder_gradient + decoder_gradient + direct_decoder_gradient)
    or encoder_gradient <= 0
    or decoder_gradient <= 0
    or direct_decoder_gradient <= 0
):
    raise RuntimeError("Gradient preflight failed")
if z_preflight_encoded.shape != x_preflight.shape or x_preflight_reconstructed.shape != x_preflight.shape:
    raise RuntimeError("Model output shape preflight failed")

for label, values in [
    ("encoded z", z_preflight_encoded.detach()),
    ("reconstructed x", x_preflight_reconstructed.detach()),
]:
    for start in (0, 4):
        p2 = (values[:, start:start + 3] ** 2).sum(dim=1)
        expected_energy = torch.sqrt(torch.clamp(p2 + MUON_MASS ** 2, min=EPS))
        maximum_error = float((values[:, start + 3] - expected_energy).abs().max().cpu())
        if maximum_error > 1.0e-5:
            raise RuntimeError(f"{label} mass-shell preflight failed: {maximum_error}")

print("Gradient preflight passed:", encoder_gradient, decoder_gradient, direct_decoder_gradient)
print("Mass-shell and shape preflight passed.")


## Training Stage 1


Warm up the residual transport deterministically with strong encoder and decoder direction anchors. Both networks are updated.


## Training Stage 2


Turn on the conditional Gaussian core response, reduce the anchors, and begin direct $D(z)$ distribution matching.


## Training Stage 3


Train the stochastic encoder and decoder jointly, introduce part of the heavy-tail response, and activate the generic mass-shape terms.


## Training Stage 4


Use the full stochastic response with denser sliced-Wasserstein projections and fixed training-quantile occupancy constraints.


## Training Stage 5


Polish the joint response at a low learning rate while preserving latent alignment and cycle reconstruction.


## Run All Training Stages


The five configurations above are executed sequentially. Each stage restores its own best validation checkpoint before the following stage.


In [ ]:
def train_all_stages():
    history = []
    global_best_score = math.inf
    global_epoch = 0

    for stage_index, stage in enumerate(ACTIVE_CONFIG["stages"]):
        print("\n" + "=" * 80)
        print(stage["name"], stage)
        set_trainable(
            model,
            encoder=not stage["freeze_encoder"],
            decoder=not stage["freeze_decoder"],
        )
        model.set_noise_multipliers(
            stage.get("core_noise_multiplier", 1.0),
            stage.get("tail_noise_multiplier", 1.0),
        )
        print(
            "Noise multipliers:",
            stage.get("core_noise_multiplier", 1.0),
            stage.get("tail_noise_multiplier", 1.0),
        )
        trainable_parameters = [
            parameter for parameter in model.parameters() if parameter.requires_grad
        ]
        optimizer = torch.optim.Adam(trainable_parameters, lr=stage["lr"])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(1, stage["epochs"])
        )

        stage_best_score = math.inf
        stage_best_path = OUTPUT_DIR / f"best_{stage['name']}.pt"
        epochs_without_improvement = 0

        for local_epoch in range(1, stage["epochs"] + 1):
            global_epoch += 1
            started = time.time()
            train_metrics = train_epoch(
                model,
                optimizer,
                x_source,
                z_source,
                x_loss_function,
                z_loss_function,
                stage,
                global_epoch,
            )
            scheduler.step()

            should_evaluate = (
                local_epoch == 1
                or local_epoch == stage["epochs"]
                or local_epoch % int(ACTIVE_CONFIG["eval_every"]) == 0
            )
            validation = None
            if should_evaluate:
                validation = evaluate_model(
                    model,
                    x_validation_fixed,
                    z_validation_fixed,
                    x_loss_function,
                    z_loss_function,
                    min(int(stage["num_slices"]), 512),
                )

            row = {
                "global_epoch": global_epoch,
                "stage_index": stage_index,
                "stage": stage["name"],
                "stage_epoch": local_epoch,
                "lr": optimizer.param_groups[0]["lr"],
                "seconds": time.time() - started,
                **{f"train_{key}": value for key, value in train_metrics.items()},
            }
            if validation is not None:
                row.update({f"validation_{key}": value for key, value in validation.items()})
            history.append(row)
            write_history(history)

            if validation is not None:
                score = validation["score"]
                improved_stage = score < stage_best_score
                improved_global = score < global_best_score

                if improved_stage:
                    stage_best_score = score
                    epochs_without_improvement = 0
                    save_checkpoint(
                        stage_best_path,
                        model,
                        optimizer,
                        stage,
                        stage_index,
                        global_epoch,
                        validation,
                        history,
                    )
                else:
                    epochs_without_improvement += int(ACTIVE_CONFIG["eval_every"])

                if improved_global:
                    global_best_score = score
                    save_checkpoint(
                        OUTPUT_DIR / "best_model.pt",
                        model,
                        optimizer,
                        stage,
                        stage_index,
                        global_epoch,
                        validation,
                        history,
                    )

                save_checkpoint(
                    OUTPUT_DIR / "last_model.pt",
                    model,
                    optimizer,
                    stage,
                    stage_index,
                    global_epoch,
                    validation,
                    history,
                )

                print(
                    f"epoch {global_epoch:4d} | "
                    f"train={train_metrics['loss']:.5f} | "
                    f"val score={score:.5f} "
                    f"sim={validation['x_sim']:.5f} "
                    f"z={validation['z_prior']:.5f} "
                    f"reco={validation['x_reconstruction']:.5f} "
                    f"mass bands={validation['mass_quantile_shape']:.4f} "
                    f"tail CDF={validation['mass_tail_cdf']:.4f} "
                    f"spacing={validation['mass_quantile_spacing']:.4f} "
                    f"fixed bins={validation['mass_fixed_bin_occupancy']:.4f} "
                    f"tail rel={validation['mass_tail_relative_error']:.3f} "
                    f"tails=({validation['mass_low_fraction']:.3f}, "
                    f"{validation['mass_high_fraction']:.3f}) | "
                    f"{row['seconds']:.1f}s"
                )

                patience = stage["patience"]
                if patience is not None and epochs_without_improvement >= patience:
                    print(f"Early stopping {stage['name']} after {local_epoch} epochs")
                    break

        if not stage_best_path.exists():
            raise RuntimeError(f"No finite checkpoint was saved for {stage['name']}")
        stage_checkpoint = load_checkpoint(stage_best_path)
        model.load_state_dict(stage_checkpoint["model_state_dict"])
        print(
            f"Restored best {stage['name']} checkpoint "
            f"(score={stage_checkpoint['validation']['score']:.6f})"
        )

    best_checkpoint = load_checkpoint(OUTPUT_DIR / "best_model.pt")
    if best_checkpoint["common_config_sha256"] != COMMON_CONFIG_HASH:
        raise RuntimeError("Checkpoint/config hash mismatch")
    model.load_state_dict(best_checkpoint["model_state_dict"])
    selected_stage = best_checkpoint["stage"]
    model.set_noise_multipliers(
        selected_stage.get("core_noise_multiplier", 1.0),
        selected_stage.get("tail_noise_multiplier", 1.0),
    )
    set_trainable(model, encoder=True, decoder=True)
    print("\nLoaded global best checkpoint:", best_checkpoint["validation"])
    return history, best_checkpoint



In [ ]:
if RUN_TRAINING:
    history, best_checkpoint = train_all_stages()
else:
    checkpoint_path = (
        Path(MANUAL_CHECKPOINT_PATH).expanduser()
        if MANUAL_CHECKPOINT_PATH
        else OUTPUT_DIR / "best_model.pt"
    )
    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Checkpoint does not exist: {checkpoint_path}. "
            "Upload the v12 best_model.pt and set MANUAL_CHECKPOINT_PATH "
            "to its exact Kaggle path, or run fresh v12 training."
        )
    best_checkpoint = load_checkpoint(checkpoint_path)
    checkpoint_hash = best_checkpoint.get("common_config_sha256")
    if checkpoint_hash != COMMON_CONFIG_HASH:
        raise RuntimeError(
            "Checkpoint/config hash mismatch: this is not compatible with v12. "
            f"checkpoint={checkpoint_hash}, expected={COMMON_CONFIG_HASH}"
        )
    selected_stage = best_checkpoint.get("stage", {})
    allowed_v12_stages = {stage["name"] for stage in ACTIVE_CONFIG["stages"]}
    checkpoint_stage_name = selected_stage.get("name")
    if checkpoint_stage_name not in allowed_v12_stages:
        raise RuntimeError(
            f"Checkpoint stage {checkpoint_stage_name!r} is not a v12 stage. "
            "Do not load the v13 encoder-refinement checkpoint."
        )
    model.load_state_dict(best_checkpoint["model_state_dict"])
    model.set_noise_multipliers(
        selected_stage.get("core_noise_multiplier", 1.0),
        selected_stage.get("tail_noise_multiplier", 1.0),
    )
    history = best_checkpoint.get("history", [])
    set_trainable(model, encoder=True, decoder=True)
    print("Loaded v12 checkpoint:", checkpoint_path)
    print("Selected v12 stage:", checkpoint_stage_name)
    print("Checkpoint validation:", best_checkpoint["validation"])


## Plot Loss History


History dictionary and validation curves:


In [ ]:
if history:
    epochs = [row["global_epoch"] for row in history]
    figure, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(epochs, [row["train_loss"] for row in history], label="train total")
    axes[0].set_yscale("log")
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("loss")
    axes[0].grid(alpha=0.25)
    axes[0].legend()

    evaluated = [row for row in history if "validation_score" in row]
    axes[1].plot(
        [row["global_epoch"] for row in evaluated],
        [row["validation_x_sim"] for row in evaluated],
        label="D(z) vs x",
    )
    axes[1].plot(
        [row["global_epoch"] for row in evaluated],
        [row["validation_z_prior"] for row in evaluated],
        label="E(x) vs z",
    )
    axes[1].plot(
        [row["global_epoch"] for row in evaluated],
        [row["validation_x_reconstruction"] for row in evaluated],
        label="D(E(x)) vs x",
    )
    axes[1].set_yscale("log")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("dimensionless validation loss")
    axes[1].grid(alpha=0.25)
    axes[1].legend()
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / "training_history.png", dpi=160)
    plt.show()


# Show Results


The primary simulator result is $D(z)$ compared with $x$. $D(E(x))$ is cycle reconstruction, and $E(x)$ compared with $z$ checks latent-space alignment.


## Get Results


In [ ]:
@torch.inference_mode()
def decode_validation_in_chunks(values, draws, batch_size=8192):
    outputs = []
    model.eval()
    with fixed_evaluation_rng(ACTIVE_CONFIG["seed"] + 515151):
        for _ in range(int(draws)):
            draw = []
            for start in range(0, len(values), batch_size):
                batch = torch.from_numpy(
                    np.ascontiguousarray(values[start:start + batch_size], dtype=np.float32)
                ).to(DEVICE)
                draw.append(model.decode(batch).cpu().numpy())
            outputs.append(np.concatenate(draw, axis=0))
    return np.concatenate(outputs, axis=0)



In [ ]:
@torch.inference_mode()
def encode_validation_in_chunks(values, batch_size=8192):
    outputs = []
    model.eval()
    with fixed_evaluation_rng(ACTIVE_CONFIG["seed"] + 616161):
        for start in range(0, len(values), batch_size):
            batch = torch.from_numpy(
                np.ascontiguousarray(
                    values[start:start + batch_size], dtype=np.float32
                )
            ).to(DEVICE)
            outputs.append(model.encode(batch).cpu().numpy())
    return np.concatenate(outputs, axis=0)



In [ ]:
def validation_mass_shape_report(model, x_values, z_values, draws=4):
    truth_mass = invariant_mass_np(x_values)
    simulated_mass = invariant_mass_np(
        decode_validation_in_chunks(z_values, draws=draws)
    )
    theory_mass = invariant_mass_np(z_values)
    encoded_mass = invariant_mass_np(encode_validation_in_chunks(x_values))
    repeated_theory_mass = np.tile(theory_mass, int(draws))
    q05, q95 = np.quantile(truth_mass, [0.05, 0.95])
    tail_fraction_report = {}
    for quantile in ACTIVE_CONFIG["loss"]["mass_tail_quantiles"]:
        lower_threshold, upper_threshold = np.quantile(
            truth_mass, [quantile, 1.0 - quantile]
        )
        lower_fraction = float(np.mean(simulated_mass < lower_threshold))
        upper_fraction = float(np.mean(simulated_mass > upper_threshold))
        tail_fraction_report[f"q{100 * quantile:g}"] = {
            "target_per_tail": float(quantile),
            "simulation_lower_fraction": lower_fraction,
            "simulation_upper_fraction": upper_fraction,
            "mean_relative_error": float(
                0.5
                * (
                    abs(lower_fraction - quantile)
                    + abs(upper_fraction - quantile)
                )
                / quantile
            ),
        }

    # Statistically meaningful residual bins: at least 1000 unique CMS events
    # per bin. These edges are quantiles, not J/psi-specific mass targets.
    minimum_truth_per_bin = 1000
    adaptive_bin_count = max(
        10,
        min(60, len(truth_mass) // minimum_truth_per_bin),
    )
    adaptive_edges = np.unique(
        np.quantile(truth_mass, np.linspace(0.0, 1.0, adaptive_bin_count + 1))
    )
    truth_counts, _ = np.histogram(truth_mass, bins=adaptive_edges)
    simulation_counts, _ = np.histogram(simulated_mass, bins=adaptive_edges)
    truth_fraction = truth_counts / truth_counts.sum()
    simulation_fraction = simulation_counts / simulation_counts.sum()
    adaptive_residual = (
        simulation_fraction - truth_fraction
    ) / np.clip(truth_fraction, 1.0e-12, None)

    # Conservative counting uncertainty: repeated decoder draws do not
    # create new independent theory events, so use len(z_values), not the
    # larger number of decoded rows, as the simulation effective size.
    truth_variance = (
        truth_fraction * (1.0 - truth_fraction) / max(len(truth_mass), 1)
    )
    simulation_variance = (
        simulation_fraction
        * (1.0 - simulation_fraction)
        / max(len(z_values), 1)
    )
    safe_truth_fraction = np.clip(truth_fraction, 1.0e-12, None)
    residual_stat_error = np.sqrt(
        simulation_variance / safe_truth_fraction**2
        + simulation_fraction**2
        * truth_variance
        / safe_truth_fraction**4
    )

    quantile_grid = np.linspace(0.0, 1.0, 4097)
    mass_w1 = np.mean(
        np.abs(
            np.quantile(truth_mass, quantile_grid)
            - np.quantile(simulated_mass, quantile_grid)
        )
    )
    latent_quantile_grid = np.linspace(0.0, 1.0, 4097)
    latent_mass_w1 = np.mean(
        np.abs(
            np.quantile(theory_mass, latent_quantile_grid)
            - np.quantile(encoded_mass, latent_quantile_grid)
        )
    )
    decoder_mass_correlation = np.corrcoef(
        repeated_theory_mass, simulated_mass
    )[0, 1]
    report = {
        "unique_cms_validation_events": int(len(truth_mass)),
        "simulated_validation_draws": int(draws),
        "cms_mass_q05_gev": float(q05),
        "cms_mass_q95_gev": float(q95),
        "simulation_mass_w1_gev": float(mass_w1),
        "latent_mass_w1_gev": float(latent_mass_w1),
        "decoder_input_output_mass_correlation": float(
            decoder_mass_correlation
        ),
        "simulation_low_tail_fraction": float(np.mean(simulated_mass < q05)),
        "simulation_high_tail_fraction": float(np.mean(simulated_mass > q95)),
        "target_fraction_per_tail": 0.05,
        "tail_fraction_report": tail_fraction_report,
        "adaptive_residual_bin_count": int(len(adaptive_residual)),
        "maximum_absolute_adaptive_residual": float(np.max(np.abs(adaptive_residual))),
        "percent_bins_within_5_percent": float(
            100.0 * np.mean(np.abs(adaptive_residual) < 0.05)
        ),
    }
    with (OUTPUT_DIR / "validation_mass_shape_metrics.json").open(
        "w", encoding="utf-8"
    ) as handle:
        json.dump(report, handle, indent=2, sort_keys=True)
    print(json.dumps(report, indent=2))

    low, high = DATA_CONFIG["mass_plot_range"]
    width = DATA_CONFIG["mass_bin_width"]
    display_edges = np.arange(low, high + 0.5 * width, width)
    figure, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True)
    for axis in axes:
        axis.hist(
            truth_mass,
            bins=display_edges,
            density=True,
            histtype="step",
            linewidth=1.8,
            label="CMS x validation",
        )
        axis.hist(
            simulated_mass,
            bins=display_edges,
            density=True,
            histtype="step",
            linewidth=1.8,
            label="OTUS D(z validation)",
        )
        axis.axvline(q05, color="gray", linestyle=":", linewidth=1)
        axis.axvline(q95, color="gray", linestyle=":", linewidth=1)
        axis.set_xlabel("m(mu mu) [GeV]")
        axis.grid(alpha=0.25)
    axes[0].set_ylabel("normalized density")
    axes[0].set_title("Full-validation mass")
    axes[0].legend()
    axes[1].set_yscale("log")
    axes[1].set_title("Full-validation mass (log scale)")
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / "validation_mass_shape.png", dpi=180)
    plt.show()

    centers = 0.5 * (adaptive_edges[:-1] + adaptive_edges[1:])
    half_width = 0.5 * (adaptive_edges[1:] - adaptive_edges[:-1])
    figure, axis = plt.subplots(figsize=(10, 4))
    axis.errorbar(
        centers,
        adaptive_residual,
        xerr=half_width,
        yerr=residual_stat_error,
        fmt=".",
        color="black",
        label="shape residual ± conservative counting error",
    )
    axis.axhline(0.0, color="gray", linewidth=1)
    axis.axhline(0.05, color="red", linestyle=":", linewidth=1)
    axis.axhline(-0.05, color="red", linestyle=":", linewidth=1)
    axis.set_xlim(low, high)
    axis.set_xlabel("m(mu mu) [GeV]")
    axis.set_ylabel("(OTUS-CMS)/CMS")
    axis.set_title("Full-validation adaptive-bin residual")
    axis.grid(alpha=0.25)
    axis.legend()
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / "validation_mass_residual.png", dpi=180)
    plt.show()

    figure, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    axes[0].hist(
        theory_mass, bins=display_edges, density=True, histtype="step",
        linewidth=1.8, label="theory z validation",
    )
    axes[0].hist(
        encoded_mass, bins=display_edges, density=True, histtype="step",
        linewidth=1.8, label="E(x validation)",
    )
    axes[0].set_xlim(low, high)
    axes[0].set_xlabel("m(mu mu) [GeV]")
    axes[0].set_ylabel("normalized density")
    axes[0].set_title("Latent-space mass support")
    axes[0].grid(alpha=0.25)
    axes[0].legend()

    axes[1].hexbin(
        repeated_theory_mass, simulated_mass, gridsize=70, bins="log",
        mincnt=1, cmap="viridis",
    )
    axes[1].plot([low, high], [low, high], linestyle=":", color="white")
    axes[1].set_xlim(low, high)
    axes[1].set_ylim(low, high)
    axes[1].set_xlabel("input m(z) [GeV]")
    axes[1].set_ylabel("output m(D(z)) [GeV]")
    axes[1].set_title("Decoder mass transport")
    axes[1].grid(alpha=0.15)
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / "validation_mass_transport.png", dpi=180)
    plt.show()
    return report



In [ ]:
validation_mass_report = validation_mass_shape_report(
    model,
    x_validation,
    z_validation,
    draws=ACTIVE_CONFIG["loader"]["validation_draws"],
)


## Inspect Principal Axes Matching On Validation Data


Raw-coordinate matching is included in the validation `raw_swd` and `raw_marginal_w1` terms. The full coordinate plots are produced in the testing section after the checkpoint is frozen.


### z-space


$E(x_{\rm validation})$ is compared with the independently sampled MadGraph prior.


### x-space


$D(z_{\rm validation})$ and $D(E(x_{\rm validation}))$ are compared with CMS.


## Inspect Random Axes Matching On Validation Data


The `sliced_wasserstein` validation terms project the raw and derived features onto fixed random axes, directly paralleling the random-axis checks in the original notebook.


## Inspect Derived Quantity Matching On Validation Data


### Dimuon invariant mass


The validation report above contains linear- and log-scale mass plots, adaptive residuals, $E(x)$ latent mass, and the decoder mass-transport hexbin plot.


## Inspect the relation between momenta of the $\mu^-$ and $\mu^+$ through the mappings


The finite both-muon `direction_anchor` replaces the original first-lepton cosine anchor. Pair-$p_T$, $\Delta\phi$, and $\Delta\eta$ are also included in the physics-feature validation.


## Inspect 2D Qualities of the Results on Validation Data


The decoder mass-transport hexbin is saved as `validation_mass_transport.png`. Additional observable correlations are evaluated on the untouched test set below.


## Inspect Mappings with Transport Plots on Validation Data


The mass-transport plot displays the eventwise relation between input $m(z)$ and output $m(D(z))$. This is the v12 analogue of the original transport-plan diagnostics.


# Save Trained Model


Training saves `best_model.pt`, the best checkpoint for every stage, `last_model.pt`, `history.csv`, the resolved configuration, normalization statistics, data-split indices, optimizer state, and random-number states.


# Evaluate Trained Model on Testing Data


Set `RUN_FINAL_TEST=True` only after the model and checkpoint-selection procedure have been frozen. The test arrays are not used during training or model selection.


In [ ]:
@torch.inference_mode()
def apply_in_chunks(function, values, batch_size=8192):
    outputs = []
    model.eval()
    for start in range(0, len(values), batch_size):
        batch = torch.from_numpy(
            np.ascontiguousarray(values[start:start + batch_size], dtype=np.float32)
        ).to(DEVICE)
        outputs.append(function(batch).cpu().numpy())
    return np.concatenate(outputs, axis=0)



In [ ]:
def exact_w1(a, b):
    if HAVE_SCIPY:
        return float(wasserstein_distance(a, b))
    quantiles = np.linspace(0.0, 1.0, max(len(a), len(b)), endpoint=True)
    return float(np.mean(np.abs(np.quantile(a, quantiles) - np.quantile(b, quantiles))))



In [ ]:
def one_dimensional_metrics(truth, prediction):
    metrics = {"w1": exact_w1(truth, prediction)}
    if HAVE_SCIPY:
        result = ks_2samp(truth, prediction)
        metrics["ks"] = float(result.statistic)
        metrics["ks_pvalue"] = float(result.pvalue)
    return metrics



In [ ]:
def normalized_histogram_residual(truth, prediction, edges, min_truth_count=20):
    truth_counts, _ = np.histogram(truth, bins=edges)
    prediction_counts, _ = np.histogram(prediction, bins=edges)
    truth_density = truth_counts / max(1, truth_counts.sum())
    prediction_density = prediction_counts / max(1, prediction_counts.sum())
    valid = (truth_counts >= min_truth_count) & (truth_density > 0)
    residual = np.full(len(edges) - 1, np.nan)
    residual[valid] = (
        prediction_density[valid] - truth_density[valid]
    ) / truth_density[valid]
    relative_stat_error = np.full(len(edges) - 1, np.nan)
    stat_mask = valid & (prediction_counts > 0)
    relative_stat_error[stat_mask] = np.sqrt(
        1.0 / truth_counts[stat_mask] + 1.0 / prediction_counts[stat_mask]
    )
    return {
        "truth_counts": truth_counts,
        "prediction_counts": prediction_counts,
        "truth_density": truth_density,
        "prediction_density": prediction_density,
        "valid": valid,
        "residual": residual,
        "relative_stat_error": relative_stat_error,
    }



In [ ]:
if RUN_FINAL_TEST:
    seed_everything(ACTIVE_CONFIG["seed"] + 900001)
    z_decoded_test = apply_in_chunks(model.decode, z_test)
    x_encoded_test = apply_in_chunks(model.encode, x_test)
    x_reconstructed_test = apply_in_chunks(model.decode, x_encoded_test)

    np.savez_compressed(
        OUTPUT_DIR / "test_outputs.npz",
        x_test=x_test,
        z_test=z_test,
        z_decoded=z_decoded_test,
        x_encoded=x_encoded_test,
        x_reconstructed=x_reconstructed_test,
    )

    test_metrics = {}
    for name, truth, prediction in [
        ("simulation_Dz", invariant_mass_np(x_test), invariant_mass_np(z_decoded_test)),
        ("latent_Ex", invariant_mass_np(z_test), invariant_mass_np(x_encoded_test)),
        ("reconstruction_DEx", invariant_mass_np(x_test), invariant_mass_np(x_reconstructed_test)),
    ]:
        test_metrics[name] = one_dimensional_metrics(truth, prediction)

    with (OUTPUT_DIR / "test_mass_metrics.json").open("w", encoding="utf-8") as handle:
        json.dump(test_metrics, handle, indent=2, sort_keys=True)

    print(json.dumps(test_metrics, indent=2))
else:
    print("Final test disabled. Set OTUS_RUN_FINAL_TEST=1 only after freezing the study.")


## Inspect Derived Quantity Matching On Testing Data


### Dimuon invariant mass


In [ ]:
if RUN_FINAL_TEST:
    mass_x = invariant_mass_np(x_test)
    mass_sim = invariant_mass_np(z_decoded_test)
    mass_reco = invariant_mass_np(x_reconstructed_test)
    mass_z = invariant_mass_np(z_test)
    mass_encoded = invariant_mass_np(x_encoded_test)

    low, high = DATA_CONFIG["mass_plot_range"]
    width = DATA_CONFIG["mass_bin_width"]
    edges = np.arange(low, high + 0.5 * width, width)
    centers = 0.5 * (edges[:-1] + edges[1:])
    residual_data = normalized_histogram_residual(mass_x, mass_sim, edges)

    figure, (top, bottom) = plt.subplots(
        2, 1, figsize=(10, 8), sharex=True,
        gridspec_kw={"height_ratios": [3, 1]},
    )
    top.hist(mass_x, bins=edges, density=True, histtype="step", linewidth=2.0, label="CMS Data")
    top.hist(mass_sim, bins=edges, density=True, histtype="step", linewidth=2.0, label="OTUS")
    top.hist(mass_reco, bins=edges, density=True, histtype="step", linewidth=1.4, linestyle="--", label=r'$x \to \tilde{z} \to \tilde{x}$')
    top.set_ylabel("normalized density")
    top.legend()
    top.grid(alpha=0.25)

    valid = residual_data["valid"]
    bottom.errorbar(
        centers[valid],
        residual_data["residual"][valid],
        yerr=residual_data["relative_stat_error"][valid],
        fmt=".",
        markersize=4,
        color="black",
        label="shape residual ± counting error",
    )
    bottom.axhline(0.0, color="gray", linewidth=1)
    bottom.axhline(0.01, color="red", linestyle=":", linewidth=1)
    bottom.axhline(-0.01, color="red", linestyle=":", linewidth=1)
    bottom.set_xlabel("m(mu mu) [GeV]")
    bottom.set_ylabel("(OTUS-data)/data")
    bottom.grid(alpha=0.25)
    bottom.legend(fontsize=8)
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / "test_simulation_mass.png", dpi=180)
    plt.show()

    figure, axis = plt.subplots(figsize=(10, 5))
    axis.hist(mass_z, bins=edges, density=True, histtype="step", linewidth=2, label="theory z test")
    axis.hist(mass_encoded, bins=edges, density=True, histtype="step", linewidth=2, label="E(x test)")
    axis.set_xlabel("m(mu mu) [GeV]")
    axis.set_ylabel("normalized density")
    axis.legend()
    axis.grid(alpha=0.25)
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / "test_latent_mass.png", dpi=180)
    plt.show()


## Inspect Principal and Derived Quantities on Testing Data


In [ ]:
def plotting_observables(values):
    features = physics_features_np(values)
    return {
        "m(mu mu) [GeV]": np.exp(features[:, 0]),
        "pair pT [GeV]": np.exp(features[:, 1]),
        "mu- pT [GeV]": np.exp(features[:, 2]),
        "mu+ pT [GeV]": np.exp(features[:, 3]),
        "mu- eta": features[:, 4],
        "mu+ eta": features[:, 5],
        "pair rapidity": features[:, 6],
        "cos delta-phi": features[:, 7],
        "delta-eta": features[:, 9],
    }



In [ ]:
if RUN_FINAL_TEST:
    truth_observables = plotting_observables(x_test)
    simulation_observables = plotting_observables(z_decoded_test)
    figure, axes = plt.subplots(3, 3, figsize=(15, 12))
    observable_metrics = {}
    for axis, name in zip(axes.flat, truth_observables):
        truth = truth_observables[name]
        prediction = simulation_observables[name]
        combined = np.concatenate([truth, prediction])
        plot_low, plot_high = np.quantile(combined, [0.005, 0.995])
        bins = np.linspace(plot_low, plot_high, 61)
        axis.hist(truth, bins=bins, density=True, histtype="step", linewidth=1.6, label="CMS")
        axis.hist(prediction, bins=bins, density=True, histtype="step", linewidth=1.6, label="D(z)")
        axis.set_title(name)
        axis.grid(alpha=0.2)
        observable_metrics[name] = one_dimensional_metrics(truth, prediction)
    axes.flat[0].legend()
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / "test_simulation_observables.png", dpi=160)
    plt.show()

    with (OUTPUT_DIR / "test_observable_metrics.json").open("w", encoding="utf-8") as handle:
        json.dump(observable_metrics, handle, indent=2, sort_keys=True)


## Inspect Stochasticity of Decoder Mapping on Testing Data


Fix each truth event and decode it repeatedly. A genuine stochastic detector model should produce a nonzero, kinematics-dependent spread without becoming unrealistically broad.


In [ ]:
if RUN_FINAL_TEST:
    # Fixed-z stochasticity diagnostic. This checks whether the decoder uses its
    # noise input for mass and for individual-muon kinematics.
    rng = np.random.default_rng(ACTIVE_CONFIG["seed"] + 777)
    fixed_count = min(256, len(z_test))
    fixed_indices = rng.choice(len(z_test), size=fixed_count, replace=False)
    fixed_z = torch.from_numpy(np.ascontiguousarray(z_test[fixed_indices])).to(DEVICE)

    repeated = []
    model.eval()
    with torch.inference_mode():
        for _ in range(32):
            decoded = model.decode(fixed_z).cpu().numpy()
            features = physics_features_np(decoded)
            phi_minus = np.arctan2(decoded[:, 1], decoded[:, 0])
            phi_plus = np.arctan2(decoded[:, 5], decoded[:, 4])
            repeated.append(
                np.stack(
                    [
                        np.exp(features[:, 0]),
                        np.exp(features[:, 2]),
                        np.exp(features[:, 3]),
                        features[:, 4],
                        features[:, 5],
                        phi_minus,
                        phi_plus,
                    ],
                    axis=1,
                )
            )
    repeated = np.stack(repeated, axis=0)

    # Wrap phi differences around the first draw before taking their width.
    phi_reference = repeated[0:1, :, 5:7]
    repeated[:, :, 5:7] = np.arctan2(
        np.sin(repeated[:, :, 5:7] - phi_reference),
        np.cos(repeated[:, :, 5:7] - phi_reference),
    )
    per_event_width = np.std(repeated, axis=0)
    names = [
        "mass_gev",
        "mu_minus_pt_gev",
        "mu_plus_pt_gev",
        "mu_minus_eta",
        "mu_plus_eta",
        "mu_minus_phi_rad",
        "mu_plus_phi_rad",
    ]
    stochasticity = {
        "fixed_events": int(fixed_count),
        "draws_per_event": 32,
        "median_per_event_std": {
            name: float(np.median(per_event_width[:, index]))
            for index, name in enumerate(names)
        },
        "mean_per_event_std": {
            name: float(np.mean(per_event_width[:, index]))
            for index, name in enumerate(names)
        },
        "mass_fraction_below_1e-5_gev": float(
            np.mean(per_event_width[:, 0] < 1.0e-5)
        ),
    }
    print(json.dumps(stochasticity, indent=2))
    with (OUTPUT_DIR / "stochasticity.json").open("w", encoding="utf-8") as handle:
        json.dump(stochasticity, handle, indent=2, sort_keys=True)

    if stochasticity["mass_fraction_below_1e-5_gev"] > 0.90:
        print("WARNING: decoder stochasticity appears collapsed; do not claim a learned resolution model.")
